In [1]:
import pandas as pd
from transformers import PegasusForConditionalGeneration, PegasusTokenizer
import torch


/Users/nirashanelki/Documents/Works/Research/codes/factual_inconsistency_validate_xai/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/nirashanelki/Documents/Works/Research/codes/factual_inconsistency_validate_xai/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/nirashanelki/Documents/Works/Research/codes/factual_inconsistency_validate_xai/.venv/lib/python3.9/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/Users/ni

In [2]:
model_path = "../model/pegasus_xsum_model"
tokenizer_path = "../model/pegasus_xsum_tokenizer"


device = "mps" if torch.backends.mps.is_available() else "cpu"

tokenizer = PegasusTokenizer.from_pretrained(tokenizer_path)
model = PegasusForConditionalGeneration.from_pretrained(model_path).to(device)


In [3]:
print(device)

mps


In [4]:
data = pd.read_csv("../data/xsum_test.csv")

data.head()

,document,summary,id
0,"Prison Link Cymru had 1,099 referrals in 2015-...","There is a ""chronic"" need for more housing for...",38264402
1,Officers searched properties in the Waterfront...,"A man has appeared in court after firearms, am...",34227252
2,"Jordan Hill, Brittany Covington and Tesfaye Co...",Four people accused of kidnapping and torturin...,38537698
3,The 48-year-old former Arsenal goalkeeper play...,West Brom have appointed Nicky Hammond as tech...,36175342
4,Restoring the function of the organ - which he...,The pancreas can be triggered to regenerate it...,39070183


In [5]:
print(len(data))

print(data.isnull().sum())

data = data.dropna().reset_index(drop=True)
print(len(data))

11334
document    1
summary     0
id          0
dtype: int64
11333


In [6]:
data_test_small = data.iloc[:100]

test_data_1 = data_test_small.copy()

In [7]:
# Generating summaries and create a new dtaframe column
for index, row in data_test_small.iterrows():
    article = row['document']
    inputs = tokenizer(article, truncation=True, padding='longest', return_tensors="pt").to(device)
    summary_ids = model.generate(inputs['input_ids'], num_beams=4, early_stopping=True)
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    test_data_1.at[index, 'generated_summary'] = summary

In [8]:
test_data_1.head()

,document,summary,id,generated_summary
0,"Prison Link Cymru had 1,099 referrals in 2015-...","There is a ""chronic"" need for more housing for...",38264402,"More than 1,000 homeless people were referred ..."
1,Officers searched properties in the Waterfront...,"A man has appeared in court after firearms, am...",34227252,A man has been charged after firearms and mone...
2,"Jordan Hill, Brittany Covington and Tesfaye Co...",Four people accused of kidnapping and torturin...,38537698,Four black people accused of torturing a menta...
3,The 48-year-old former Arsenal goalkeeper play...,West Brom have appointed Nicky Hammond as tech...,36175342,West Bromwich Albion have appointed Reading di...
4,Restoring the function of the organ - which he...,The pancreas can be triggered to regenerate it...,39070183,"The ""fasting-mimicking"" diet can regenerate th..."


In [9]:
# save the dataframe with generated summaries to a new CSV file
test_data_1.to_csv("../data/test/xsum_test_with_generated_summaries_1.csv", index=False)

## Model spef

In [3]:
test_data_1 = pd.read_csv("../data/test/xsum_test_with_generated_summaries_1.csv")

In [5]:
# inferencing just one article to check the output
article = test_data_1.iloc[1]["document"]
inputs = tokenizer(
    article,
    truncation=True,
    padding="longest",
    return_tensors="pt",
).to(device)
summary_ids = model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs.get("attention_mask"),
    num_beams=4,
    max_new_tokens=512,
    length_penalty=1.0,
    early_stopping=True,
    output_scores=True,
    return_dict_in_generate=True,
).sequences
summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print("Generated Summary:", summary)

/Users/nirashanelki/Documents/Works/Research/codes/factual_inconsistency_validate_xai/.venv/lib/python3.9/site-packages/transformers/generation/utils.py:1473: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed soon, in a future version. Please use and modify the model generation configuration (see https://huggingface.co/docs/transformers/generation_strategies#default-text-generation-configuration )
  warnings.warn(


Generated Summary: A man has been charged after firearms and money were seized by police in Edinburgh.
